# Elo / Chessmetrics / Hybrid Model Experiment

In [1]:
import os

# Move up one level to set the working directory to the repo root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

## Setup

In [22]:
import numpy as np
import pandas as pd

# Load the dataset
# file_path = "/mnt/data/MRegularSeasonCompactResults.csv"
# df = pd.read_csv(file_path)


# Define the reverse_loc function
def reverse_loc(WLoc):
    """Reverse the game location."""
    return {"H": "A", "A": "H", "N": "N"}.get(WLoc, WLoc)


def make_game_data_from_results(df, gender):
    """
    Transforms the input dataframe of game results into a standardized format with Team1, Team2, Scores,
    Result, and Location.

    Args:
        df (pd.DataFrame): Input dataframe with columns ['Season', 'DayNum', 'WTeamID', 'LTeamID',
                                                          'WScore', 'LScore', 'WLoc'].

    Returns:
        pd.DataFrame: Transformed dataframe with standardized team columns, scores, results, and location.
    """
    df_optimized = df.copy()

    df_optimized["gender"] = gender

    # Compute Team1 and Team2
    df_optimized["Team1"] = np.minimum(df_optimized["WTeamID"], df_optimized["LTeamID"])
    df_optimized["Team2"] = np.maximum(df_optimized["WTeamID"], df_optimized["LTeamID"])

    # Generate a unique game_id
    df_optimized["game_id"] = (
        df_optimized["Season"].astype(str)
        + "_"
        + df_optimized["DayNum"].astype(str)
        + "_"
        + df_optimized["Team1"].astype(str)
        + "_"
        + df_optimized["Team2"].astype(str)
    )

    # Compute boolean masks for Team1 == WTeamID
    team1_is_winner = df_optimized["Team1"] == df_optimized["WTeamID"]

    # Use vectorized assignment for scores
    df_optimized["Score1"] = np.where(
        team1_is_winner, df_optimized["WScore"], df_optimized["LScore"]
    )
    df_optimized["Score2"] = np.where(
        team1_is_winner, df_optimized["LScore"], df_optimized["WScore"]
    )

    # Compute result directly
    df_optimized["Result"] = team1_is_winner.astype(int)

    # Compute location using vectorized logic
    df_optimized["Loc"] = np.where(
        df_optimized["WLoc"] == "N",
        "N",
        np.where(
            team1_is_winner, df_optimized["WLoc"], df_optimized["WLoc"].map(reverse_loc)
        ),
    )

    # Select relevant columns
    df_optimized = df_optimized[
        [
            "Season",
            "DayNum",
            "Team1",
            "Team2",
            "Score1",
            "Score2",
            "Result",
            "Loc",
            "gender",
            "game_id",
        ]
    ]

    return df_optimized


# Apply the function to transform the dataset
# df_transformed = make_game_data_from_results(df)

In [8]:
from pathlib import Path

KAGGLE_DATA_PATH = Path(os.path.abspath(os.path.join(os.getcwd(), "data/kaggle")))

m_results_reg = pd.read_csv(KAGGLE_DATA_PATH / "MRegularSeasonCompactResults.csv")

In [23]:
m_results_reg_transformed = make_game_data_from_results(m_results_reg, "Men")

In [24]:
m_results_reg_transformed.head(25)

,Season,DayNum,Team1,Team2,Score1,Score2,Result,Loc,gender,game_id
0,1985,20,1228,1328,81,64,1,N,Men,1985_20_1228_1328
1,1985,25,1106,1354,77,70,1,H,Men,1985_25_1106_1354
2,1985,25,1112,1223,63,56,1,H,Men,1985_25_1112_1223
3,1985,25,1165,1432,70,54,1,H,Men,1985_25_1165_1432
4,1985,25,1192,1447,86,74,1,H,Men,1985_25_1192_1447
5,1985,25,1218,1337,79,78,1,H,Men,1985_25_1218_1337
6,1985,25,1226,1228,44,64,0,N,Men,1985_25_1226_1228
7,1985,25,1242,1268,58,56,1,N,Men,1985_25_1242_1268
8,1985,25,1133,1260,80,98,0,A,Men,1985_25_1133_1260
9,1985,25,1305,1424,97,89,1,H,Men,1985_25_1305_1424


In [33]:
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss, log_loss, accuracy_score
from sklearn.model_selection import train_test_split, KFold
from pathlib import Path
import os

# 📌 Configuration Constants
SEASON_START = 2010
SEASON_END = 2024
K_VALUES = list(range(10, 81, 10))  # Elo search range
INTERVAL_VALUES = np.linspace(5, 15, 20)  # Chessmetrics interval tuning range
HOME_COURT_ADVANTAGE = 50  # Elo bonus for home teams
MOV_SCALING = 10  # Elo MOV adjustment
DECAY_FACTOR = 0.95  # Chessmetrics opponent rating decay factor
ELO_MEAN = 1500  # Default Elo rating
KAGGLE_DATA_PATH = Path(os.path.abspath(os.path.join(os.getcwd(), "data/kaggle")))

# 📌 1️⃣ Load and Process Data
# def load_game_data():
#     """Load and process men's and women's NCAA game data."""
#     m_results = pd.read_csv(KAGGLE_DATA_PATH / "MNCAATourneyCompactResults.csv")
#     w_results = pd.read_csv(KAGGLE_DATA_PATH / "WNCAATourneyCompactResults.csv")

#     def process_results(df, gender):
#         df["gender"] = gender
#         df["game_id"] = df.apply(lambda row: f"{row.Season}_{row.WTeamID}_{row.LTeamID}", axis=1)
#         df["Team1"] = df["WTeamID"]
#         df["Team2"] = df["LTeamID"]
#         df["Score1"] = df["WScore"]
#         df["Score2"] = df["LScore"]
#         return df[["Season", "game_id", "gender", "Team1", "Team2", "Score1", "Score2"]]

#     return pd.concat([process_results(m_results, "Men"), process_results(w_results, "Women")], ignore_index=True)


# 📌 2️⃣ Compute Elo Ratings
def compute_elo(df, base_k):
    """Compute Elo ratings for teams."""
    elo_ratings = {
        team: ELO_MEAN for team in pd.concat([df["Team1"], df["Team2"]]).unique()
    }
    game_data = []

    for _, row in df.iterrows():
        team1, team2 = row["Team1"], row["Team2"]
        score1, score2 = row["Score1"], row["Score2"]
        rating1, rating2 = elo_ratings[team1], elo_ratings[team2]

        expected_win_prob = 1 / (1 + 10 ** ((rating2 - rating1) / 400))
        actual_outcome = 1 if score1 > score2 else 0
        mov = abs(score1 - score2)
        k_adjusted = base_k * (1 + np.log1p(mov) / MOV_SCALING)

        new_rating1 = rating1 + k_adjusted * (actual_outcome - expected_win_prob)
        new_rating2 = rating2 - k_adjusted * (actual_outcome - expected_win_prob)

        elo_ratings[team1], elo_ratings[team2] = new_rating1, new_rating2
        game_data.append(
            {
                "Season": row["Season"],
                "Team": team1,
                "EloRating": new_rating1,
                "gender": row["gender"],
            }
        )
        game_data.append(
            {
                "Season": row["Season"],
                "Team": team2,
                "EloRating": new_rating2,
                "gender": row["gender"],
            }
        )

    return pd.DataFrame(game_data)


# 📌 3️⃣ Compute Chessmetrics Ratings
def compute_chessmetrics(df, interval):
    """Compute Chessmetrics-style iterative ratings."""
    all_teams = set(df["Team1"]).union(
        set(df["Team2"])
    )  # Ensure all teams are included
    chess_ratings = {
        team: ELO_MEAN for team in all_teams
    }  # Initialize all teams with default rating

    for _ in range(10):  # Iteratively update ratings
        new_ratings = {}
        for team, games in df.groupby("Team1"):
            opp_ratings = [
                chess_ratings.get(opp, ELO_MEAN) for opp in games["Team2"]
            ]  # Use default rating if missing
            game_scores = [
                1 / (1 + 10 ** ((opp_score - team_score) / interval))
                for team_score, opp_score in zip(games["Score1"], games["Score2"])
            ]
            avg_opp_rating = (
                np.mean(opp_ratings) if opp_ratings else ELO_MEAN
            )  # Avoid empty list errors
            avg_game_score = (
                np.mean(game_scores) if game_scores else 0.5
            )  # Default to neutral result
            new_ratings[team] = avg_opp_rating + (10 * np.log10(avg_game_score))

        chess_ratings.update(new_ratings)  # Update ratings for next iteration

    return pd.DataFrame(
        [
            {"Season": df.Season.iloc[0], "Team": t, "ChessRating": r}
            for t, r in chess_ratings.items()
        ]
    )


# 📌 4️⃣ Create Predictive Features
def create_features(df, elo_df, chess_df):
    """Merge Elo and Chessmetrics ratings into game data and compute predictive features."""

    # Reduce elo_df to only required columns (prevent excessive memory usage)
    elo_df = elo_df[["Season", "Team", "EloRating"]].drop_duplicates()
    chess_df = chess_df[["Season", "Team", "ChessRating"]].drop_duplicates()

    # Merge Elo ratings for Team1 and Team2
    df = df.merge(
        elo_df.rename(columns={"Team": "Team1", "EloRating": "EloRating1"}),
        on=["Season", "Team1"],
        how="left",
    )
    df = df.merge(
        elo_df.rename(columns={"Team": "Team2", "EloRating": "EloRating2"}),
        on=["Season", "Team2"],
        how="left",
    )

    # Merge Chessmetrics ratings for Team1 and Team2
    df = df.merge(
        chess_df.rename(columns={"Team": "Team1", "ChessRating": "ChessRating1"}),
        on=["Season", "Team1"],
        how="left",
    )
    df = df.merge(
        chess_df.rename(columns={"Team": "Team2", "ChessRating": "ChessRating2"}),
        on=["Season", "Team2"],
        how="left",
    )

    # Compute rating differences
    df["EloDiff"] = df["EloRating1"] - df["EloRating2"]
    df["ChessDiff"] = df["ChessRating1"] - df["ChessRating2"]

    # Drop rows where any rating is missing
    df = df.dropna(subset=["EloDiff", "ChessDiff"])

    return df


# 📌 5️⃣ Train XGBoost Model
def train_xgb(df):
    """Train an XGBoost classifier to predict tournament game winners."""
    X = df[["EloDiff", "ChessDiff"]]
    y = (df["Score1"] > df["Score2"]).astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss")
    model.fit(X_train, y_train)

    preds = model.predict_proba(X_test)[:, 1]
    return {
        "Brier Score": brier_score_loss(y_test, preds),
        "Log Loss": log_loss(y_test, preds),
        "Accuracy": accuracy_score(y_test, preds > 0.5),
        "Model": model,
    }


# 📌 Run Pipeline
# df = load_game_data()
df = m_results_reg_transformed.copy()
elo_df = compute_elo(df, 30)
chess_df = compute_chessmetrics(df, 10)
feature_df = create_features(df, elo_df, chess_df)
metrics = train_xgb(feature_df)
print(metrics)

MemoryError: Unable to allocate 1.21 GiB for an array with shape (163025586,) and data type int64

In [35]:
elo_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 384994 entries, 0 to 384993
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Season     384994 non-null  int64  
 1   Team       384994 non-null  int64  
 2   EloRating  384994 non-null  float64
 3   gender     384994 non-null  object 
dtypes: float64(1), int64(2), object(1)
memory usage: 11.7+ MB


In [36]:
chess_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Season       380 non-null    int64  
 1   Team         380 non-null    int64  
 2   ChessRating  380 non-null    float64
dtypes: float64(1), int64(2)
memory usage: 9.0 KB


## 1. Data Processing

* Load and format past tournament games.
* Ensure consistency across men’s and women’s data.
* Prepare data for Elo and Chessmetrics calculations.

In [ ]:
game_data = load_game_data()

In [ ]:
game_data.head()